We're talking about Monte Carlo methods now! Exciting. Let's first go through a bit of theory about probability distribution functions, cumulative distribution functions, and think about what they mean for our problems. 

**Go to handwritten notes**

Let's implmement our first Monte Carlo program. 

Consider a slab with a beam of neutrons hitting it. The slab has a total interaction cross section of $\Sigma_t$ a thickness, and we want to run a number of particles to simulate this system. 

We will use random numbers to sample in our system and calculate a distance to collision. If the distance to collision is greater than the thickness, the particle has transmitted the slab.  

<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> How do you think we'll need to sample in this problem? 
<p></p>
Hint: Can we assume that the distribution from np.random.random() is sufficient for an exponential function?  </h2>
</div>

In [ ]:
def slab_transmission(Sig_t,thickness,N):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
        Sig_t: The total macroscopic x-section
        thickness: Width of the slab
        N: Number of neutrons to simulate
    Returns:
        transmission: The fraction of neutrons that made it through
    """
    thetas = np.random.random(N)
    x = -np.log(1-thetas)/Sig_t
    transmission = np.sum(x>thickness)/N
    #for a small number of neutrons we’ll output a little more
    if (N<=1000):
        plt.scatter(x,np.arange(N))
        plt.xlabel("Distance to collision")
        plt.ylabel("Neutron Number")
    return transmission

Note that we are using `np.random.random(N)` to get an array of random numbers. I showed this a few weeks ago, but there are *lots* of random distributions one can choose from in numpy. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#test the functionwith a small number of neutrons
Sigma_t = 2.0
thickness = 3.0
N = 1000
transmission = slab_transmission(Sigma_t, thickness, N)
print("Out of",N,"neutrons only",int(transmission*N),
    "made it through.\n The fraction that made it through was",
    transmission)

Does it make sense that this number of neutrons made it through the slab? 

Let's try to see if we run more particles in our solver.... 

In [ ]:
neuts = np.array([2000,4000,8000,16000,32000,
                  64000,128000,256e3,512e3,1024e3,2056e3])
for N in neuts:
    transmission = slab_transmission(Sigma_t, thickness, int(N))
    print("Out of",N,"neutrons only",int(transmission*N),
          "made it through.\n The fraction that made it through was",
          transmission)

As we add more and more particles to our problem, we get a better and better answer for this problem. In fact, in Monte Carlo methods we know that our error will reduce by a factor of $\frac{1}{\sqrt{N}}$


<div class="alert alert-block alert-info">
<h2> <b>Think-pair-share:</b> Do you expect to have the same values for transmission as me or people around you? Why or why not? 
</h2>
</div>

So far our particles are assuming all particles are impinging on the slab as a beam. What if we had a distribution of neutron directions? Neutrons that don't directly enter the slab may see much greater thicknesses than 3cm to transit through the slab. Let's modify our code to sample neutrons isotropically along the slab. The slab is 3cm, so the effective thickness that neutrons see is: 


$$
thickness(\phi) = \frac{3}{\cos\phi}
$$

If we sample neutrons in angle $\mu = \cos\phi$, knowing $\mu$ is within $[0,1]$, then we can get the effective distance the particle needs to transmit the slab. We can modify our input accordingly. 

- sample $\mu$
- sample a distance to collision
- check to see if the distance to collision is greater than 3/$\mu$

First, the actual answer for this is $\approx  0.000318257.$ 

In [ ]:
def slab_transmission(Sig_t,thickness,N,isotropic=False):
    """Compute the fraction of neutrons that leak through a slab
    Inputs:
        Sig_t: The total macroscopic x-section
        thickness: Width of the slab
        N: Number of neutrons to simulate
        isotropic: Are the neutrons isotropic or a beam
        isotropic: Are the neutrons isotropic or a beam
    Returns:
    transmission: The fraction of neutrons that made it through
    """
    if (isotropic):
        mu = np.random.random(N)
    else:
        mu = np.ones(N)
    thetas = np.random.random(N)
    x = -np.log(1-thetas)/Sig_t
    transmission = np.sum(x>thickness/mu)/N
    #for a small number of neutrons we’ll output a little more
    if (N<=1000):
        plt.scatter(x*mu,np.arange(N))
        plt.xlabel("Distance traveled into slab")
        plt.ylabel("Neutron Number")
    return transmission

As before let's start this with a small number of particles... 

In [ ]:
###testthe function with a small number of neutrons
Sigma_t = 2.0
thickness = 3.0
N = 1000
transmission = slab_transmission(Sigma_t, thickness, N, isotropic=True)
print("Out of",N,"neutrons only",int(transmission*N),
      "made it through.\n The fraction that made it through was",
      transmission)

... and let's see how our transmission changes as we increaese our particle count... 

In [ ]:
neuts = np.array([2000,4000,8000,16000,32000,64000,128000,256e3,512e3,1024e3,2056e3])
for N in neuts:
    transmission = slab_transmission(Sigma_t, thickness, int(N), isotropic=True)
    print("Out of",N,"neutrons only",int(transmission*N),
          "made it through.\n The fraction that made it through was",
          transmission)

We do start seeing the correct answer, but since so few neutrons get through, we need to throw a LOT to see an answer that matches theory. Here's the result if we try 10 million: 

In [ ]:
N = int(1e7)
transmission = slab_transmission(Sigma_t, thickness, N, isotropic=True)
print("Out of",N,"neutrons only",int(transmission*N),
      "made it through.\n The fraction that made it through was",
      transmission)

So far our problems are pretty easy to solve by hand. But I showed you what the transport equation looked like -- there are lots of physics we want to simulate in a real problem. For next time, start to think about how we might add in scattering to this type of problem.... 